In [ ]:
!pip install transformers datasets torch scikit-learn


In [ ]:
pip install hf_xet

In [32]:
from datasets import load_dataset

# IMDb film yorumları veri seti
dataset = load_dataset("imdb")

# Eğitim ve test verileri
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts = dataset['test']['text']
test_labels = dataset['test']['label']


In [33]:
# Eğer Dataset sütunu tipinde ise listeye çevir
train_texts = list(dataset['train']['text'])
train_labels = list(dataset['train']['label'])
test_texts = list(dataset['test']['text'])
test_labels = list(dataset['test']['label'])


In [34]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenizasyon
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


In [35]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


In [45]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
# Örnek cümleler
texts = ["I loved this movie!", "This was a terrible film."]

encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
outputs = model(**encodings)
preds = torch.argmax(outputs.logits, dim=1)

for text, pred in zip(texts, preds):
    print(f"{text} --> {'Positive' if pred==1 else 'Negative'}")
